# ML Agent — Performance Report

Visualizes the trained RL agent's backtest on the **test set** (2021-03-14 → 2026-06-30) against three baselines, plus the training curve.

**Prerequisites** (run from project root):
```bash
python -m src.agent.trainer            # train (writes data/logs/agent_training_*.jsonl)
python -m src.agent.evaluate           # backtest (writes data/backtest/*)
```
Then run all cells. Strategies: `agent` (PPO), `equal_weight`, `market_cap`, `inv_vol`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd()
while not (ROOT / 'data' / 'backtest').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent  # allow running from src/visualizations/ or project root

# Prefer the walk-forward result (continuous OOS across all windows, written by
# rolling_eval.py) over the single fixed-split backtest; fall back if not yet run.
BT = ROOT / 'data/backtest'
res_path = BT / 'walkforward_results.parquet'
met_path = BT / 'walkforward_metrics.json'
if not res_path.exists():
    res_path, met_path = BT / 'results.parquet', BT / 'metrics.json'
print(f'Source: {res_path.name}')

results = pd.read_parquet(res_path)
metrics = json.loads(met_path.read_text())

STRATEGIES = [c.removeprefix('value_') for c in results.columns if c.startswith('value_')]
weight_cols = [c for c in results.columns if c.startswith('w_')]
print(f'Backtest: {results["date"].min().date()} → {results["date"].max().date()} '
      f'({len(results)} days) | strategies: {STRATEGIES} | {len(weight_cols)} held tickers')

## 1. Metrics Comparison

The agent must beat `equal_weight` to justify its existence — that is the honest bar, not absolute Sharpe.

In [12]:
table = pd.DataFrame(metrics).T
table['cumulative_return'] = table['cumulative_return'].map('{:+.1%}'.format)
table['annualized_return'] = table['annualized_return'].map('{:+.1%}'.format)
table['max_drawdown'] = table['max_drawdown'].map('{:.1%}'.format)
table['win_rate'] = table['win_rate'].map('{:.1%}'.format)
table[['sharpe', 'sortino']] = table[['sharpe', 'sortino']].astype(float).round(3)
table.drop(columns='n_days')

,cumulative_return,annualized_return,sharpe,sortino,max_drawdown,win_rate
agent,+79.0%,+11.8%,0.574,0.869,27.4%,51.8%
equal_weight,+105.6%,+14.8%,0.708,1.069,24.8%,51.8%
market_cap,+66.1%,+10.2%,0.561,0.833,20.2%,51.6%
inv_vol,+26.2%,+4.5%,0.247,0.307,38.0%,52.2%


## 2. Portfolio Value Over Time (log scale)

In [13]:
fig = go.Figure()
for s in STRATEGIES:
    fig.add_trace(go.Scatter(x=results['date'], y=results[f'value_{s}'], name=s,
                             line=dict(width=3 if s == 'agent' else 1.5)))
fig.update_layout(title='Portfolio Value — Test Set (R$100k start)',
                  yaxis_type='log', yaxis_title='Value (R$, log)', height=500)
fig.show()

## 3. Drawdown

In [14]:
fig = go.Figure()
for s in ['agent', 'equal_weight']:
    v = results[f'value_{s}'].to_numpy()
    dd = (np.maximum.accumulate(v) - v) / np.maximum.accumulate(v)
    fig.add_trace(go.Scatter(x=results['date'], y=-dd * 100, name=s,
                             fill='tozeroy' if s == 'agent' else None))
fig.update_layout(title='Drawdown from Peak', yaxis_title='Drawdown (%)', height=450)
fig.show()

## 4. Rolling 6-Month Sharpe (is performance consistent or one lucky streak?)

In [15]:
WINDOW = 126  # ~6 trading months
fig = go.Figure()
for s in ['agent', 'equal_weight']:
    r = np.log(results[f'value_{s}'] / results[f'value_{s}'].shift(1))
    roll_sharpe = r.rolling(WINDOW).mean() / r.rolling(WINDOW).std() * np.sqrt(252)
    fig.add_trace(go.Scatter(x=results['date'], y=roll_sharpe, name=s))
fig.add_hline(y=0, line_dash='dot')
fig.update_layout(title=f'Rolling {WINDOW}d Annualized Sharpe', height=450)
fig.show()

## 5. Monthly Returns Heatmap (agent)

In [16]:
monthly = (
    results.set_index('date')['log_return']
    .resample('ME').sum()
    .pipe(lambda s: (np.exp(s) - 1) * 100)
)
hm = monthly.to_frame('ret')
hm['year'], hm['month'] = hm.index.year, hm.index.month
pivot = hm.pivot(index='year', columns='month', values='ret')
fig = go.Figure(go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index,
                           colorscale='RdYlGn', zmid=0,
                           text=np.round(pivot.values, 1), texttemplate='%{text}'))
fig.update_layout(title='Agent Monthly Returns (%)', xaxis_title='Month',
                  yaxis_title='Year', yaxis_autorange='reversed', height=420)
fig.show()

## 6. Portfolio Concentration & Top Holdings

## 6a. Full Portfolio Allocation (All Holdings + CASH)

In [ ]:
# Full allocation: top holdings + CASH + "other"
all_weights = results[weight_cols].copy()

# Handle CASH if present
if 'w_CASH' in all_weights.columns:
    cash_weight = all_weights.pop('w_CASH') * 100
else:
    cash_weight = pd.Series(0.0, index=results.index)

# Get top 15 holdings by mean weight
top_holdings = all_weights.mean().sort_values(ascending=False).head(15).index.tolist()
top_w = all_weights[top_holdings] * 100
other_w = all_weights.drop(columns=top_holdings).sum(axis=1) * 100

# Build stacked chart
fig = go.Figure()
for col in top_holdings:
    fig.add_trace(go.Scatter(x=results['date'], y=results[col] * 100,
                             name=col.removeprefix('w_'), stackgroup='alloc', mode='lines'))
if other_w.max() > 0:
    fig.add_trace(go.Scatter(x=results['date'], y=other_w,
                             name='Other (265 tickers)', stackgroup='alloc', mode='lines'))
fig.add_trace(go.Scatter(x=results['date'], y=cash_weight,
                         name='CASH (risk-free)', stackgroup='alloc', mode='lines',
                         line=dict(color='gold', width=2)))

fig.update_layout(title='Complete Portfolio Allocation Over Time (Top 15 + CASH + Other)',
                  yaxis_title='Weight (%)', height=550, hovermode='x unified')
fig.show()

In [17]:
W = results[weight_cols].to_numpy()
top10_share = np.sort(W, axis=1)[:, -10:].sum(axis=1)
n_effective = 1.0 / (W ** 2).sum(axis=1)  # inverse HHI: 'effective number of stocks'

fig = go.Figure()
fig.add_trace(go.Scatter(x=results['date'], y=top10_share * 100, name='top-10 weight share (%)'))
fig.add_trace(go.Scatter(x=results['date'], y=n_effective, name='effective # stocks', yaxis='y2'))
fig.update_layout(title='Concentration: Top-10 Share & Effective Diversification',
                  yaxis_title='Top-10 share (%)',
                  yaxis2=dict(title='Effective # stocks', overlaying='y', side='right'),
                  height=450)
fig.show()

mean_w = results[weight_cols].mean().sort_values(ascending=False).head(10)
fig = go.Figure()
for col in mean_w.index:
    fig.add_trace(go.Scatter(x=results['date'], y=results[col] * 100,
                             name=col.removeprefix('w_'), stackgroup='w'))
fig.update_layout(title='Top 10 Holdings Over Time (stacked)', yaxis_title='Weight (%)', height=450)
fig.show()

## 7. Sector Exposure Over Time

In [18]:
sectors = (
    pd.read_parquet(ROOT / 'data/processed/ml_dataset_training.parquet',
                    columns=['ticker', 'sector'])
    .drop_duplicates('ticker').set_index('ticker')['sector']
)
col_sector = {c: sectors.get(c.removeprefix('w_'), 'Unknown') for c in weight_cols}
sector_w = results[weight_cols].T.groupby(pd.Series(col_sector)).sum().T
sector_w.index = results['date']
top_sectors = sector_w.mean().sort_values(ascending=False).head(8).index

fig = go.Figure()
for s in top_sectors:
    fig.add_trace(go.Scatter(x=sector_w.index, y=sector_w[s] * 100, name=s[:35], stackgroup='s'))
fig.update_layout(title='Sector Exposure (top 8 sectors)', yaxis_title='Weight (%)', height=480)
fig.show()

## 8. Daily Return Distribution

In [19]:
r = results['log_return'] * 100
fig = go.Figure(go.Histogram(x=r, nbinsx=100))
fig.add_vline(x=r.mean(), line_dash='dash', annotation_text=f'mean {r.mean():.3f}%')
fig.update_layout(title='Agent Daily Log Returns', xaxis_title='Return (%)', height=420)
fig.show()
print(f'skew: {r.skew():.2f} | kurtosis: {r.kurtosis():.2f} | '
      f'best day: {r.max():+.2f}% | worst day: {r.min():+.2f}%')

skew: -0.16 | kurtosis: 0.75 | best day: +4.36% | worst day: -5.02%


## 9. Training Curve (validation Sharpe vs timesteps)

Reads the most recent training log. If val Sharpe rises then flattens, training converged; if it declines, early stopping should have fired.

In [20]:
logs = sorted((ROOT / 'data/logs').glob('agent_training_*.jsonl'))
if logs:
    hist = pd.read_json(logs[-1], lines=True)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist['timesteps'], y=hist['val_sharpe'],
                             mode='lines+markers', name='val Sharpe'))
    fig.add_trace(go.Scatter(x=hist['timesteps'], y=hist['val_max_drawdown'] * 100,
                             mode='lines+markers', name='val max DD (%)', yaxis='y2'))
    fig.update_layout(title=f'Training: Validation Metrics ({logs[-1].name})',
                      xaxis_title='Timesteps', yaxis_title='Sharpe',
                      yaxis2=dict(title='Max DD (%)', overlaying='y', side='right'),
                      height=450)
    fig.show()
else:
    print('No training logs found in data/logs/ — run the trainer first.')